# Addestramento di recap-projexa (Projexa)

Questo notebook addestra **recap-projexa** (Qwen3 8B, qwen3:8b), il modello che scrive **Recap cliente** e **Recap interno** quando l'AI del recap è «Projexa», sugli esempi che hai segnato come corretti nella pagina **Analisi PM › Recap** di Projexa.
Il risultato è un piccolo file **adattatore** (`recap-projexa-adapter.gguf`, 100-300 MB) da caricare nella stessa pagina (scheda *Modello sulla VM*, scegliendo **recap-projexa**). Non tocca l'Assistente PM (pm-projexa).

**Prima di iniziare**
1. Menu **Runtime → Cambia tipo di runtime → GPU T4** (gratuita) → Salva.
2. Tieni a portata il file `recap_projexa_addestramento_AAAA-MM-GG.jsonl` scaricato da Analisi PM › Recap.

**Poi** menu **Runtime → Esegui tutto**. Alla cella 2 si apre il pulsante per caricare il file .jsonl.
Tempo indicativo: 30-90 minuti (i recap sono testi lunghi). Alla fine il file .gguf si scarica da solo.

Se una cella dà errore, copia il messaggio e mandalo a chi ti segue lo sviluppo: le librerie cambiano spesso versione.


## 1. Installazione delle librerie (5-10 minuti)


In [ ]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
import torch
assert torch.cuda.is_available(), 'Nessuna GPU: Runtime > Cambia tipo di runtime > GPU T4'
print('GPU:', torch.cuda.get_device_name(0))


## 2. Carica il file degli esempi (.jsonl scaricato da Projexa)


In [ ]:
from google.colab import files
import json, collections
caricati = files.upload()
nome = next(n for n in caricati if n.endswith('.jsonl'))
esempi = [json.loads(r) for r in open(nome, encoding='utf-8') if r.strip()]
def compito(e):
    u = e['messages'][-2]['content']
    return 'appunti' if u.startswith('Questo è un pezzo') else 'recap'
print(len(esempi), 'esempi:', dict(collections.Counter(compito(e) for e in esempi)))
if len(esempi) < 30:
    print('ATTENZIONE: meno di 30 esempi, il modello imparerà poco. Meglio aspettare di averne di più.')


## 3. Modello base (Qwen3 8B, lo stesso di recap-projexa sulla VM)

I recap sono lunghi: si addestra con testi fino a 8.192 token (gli esempi più lunghi si saltano).


In [ ]:
from unsloth import FastLanguageModel
LUNGHEZZA = 8192
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen3-8B-unsloth-bnb-4bit', max_seq_length=LUNGHEZZA, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=42)


## 4. Addestramento


In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
def testo(e):
    try:
        return tokenizer.apply_chat_template(e['messages'], tokenize=False, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(e['messages'], tokenize=False)
testi = [testo(e) for e in esempi]
tenuti = [t for t in testi if len(tokenizer(t)['input_ids']) <= LUNGHEZZA]
print(f'{len(tenuti)} esempi usati, {len(testi) - len(tenuti)} saltati perché troppo lunghi')
dati = Dataset.from_dict({'text': tenuti}).shuffle(seed=42)
epoche = 3 if len(tenuti) < 150 else 2
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dati,
    args=SFTConfig(dataset_text_field='text', max_seq_length=LUNGHEZZA, per_device_train_batch_size=1,
                   gradient_accumulation_steps=8, num_train_epochs=epoche, learning_rate=1e-4,
                   warmup_steps=5, logging_steps=5, optim='adamw_8bit', fp16=True, bf16=False,
                   output_dir='uscita', report_to='none', seed=42))
trainer.train()


## 5. Prova veloce (appunti di un pezzo)


In [ ]:
FastLanguageModel.for_inference(model)
sistema = esempi[0]['messages'][0] if esempi[0]['messages'][0]['role'] == 'system' else None
prova = ([sistema] if sistema else []) + [{'role': 'user', 'content': 'Questo è un pezzo (1 di 1) della trascrizione di una riunione.\nScrivi appunti in italiano, un punto per riga.\n\nTrascrizione:\nMario Rossi: il file delle causali ve lo mandiamo entro venerdì.\nAnna Bianchi: ok, poi apriamo il ticket MySupport per la fix delle timbrature.'}]
try:
    ids = tokenizer.apply_chat_template(prova, add_generation_prompt=True, return_tensors='pt', enable_thinking=False).to('cuda')
except TypeError:
    ids = tokenizer.apply_chat_template(prova, add_generation_prompt=True, return_tensors='pt').to('cuda')
out = model.generate(input_ids=ids, max_new_tokens=150, temperature=0.3)
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))


## 6. Adattatore in formato Ollama (.gguf) e download


In [ ]:
model.save_pretrained('adattatore')
tokenizer.save_pretrained('adattatore')
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements/requirements-convert_lora_to_gguf.txt
!python llama.cpp/convert_lora_to_gguf.py adattatore --base-model-id Qwen/Qwen3-8B --outtype f16 --outfile recap-projexa-adapter.gguf
import os
print('Dimensione:', round(os.path.getsize('recap-projexa-adapter.gguf') / 1e6), 'MB')
files.download('recap-projexa-adapter.gguf')


## 7. Fatto

Carica `recap-projexa-adapter.gguf` in Projexa: **Analisi PM → Modello sulla VM**, scegli **recap-projexa** e premi **Carica e installa** (non durante una call con l'Assistente PM).
Projexa fa il backup del modello attuale, installa il nuovo e lo prova; se qualcosa non va torna da solo al precedente.
Quando sei sicuro che il nuovo è migliore, dalla stessa pagina puoi eliminare il backup.
